### TF-IDF가 중요한 이유
- 문서에 자주 나오는 단어가 항상 중요한 단어는 아니다. 
- 이 문서에는 자주 나오지만 다른 문서에는 드문 단어가 이 문서의 핵심 단어
- 예) 기자라는 단어는 어느 뉴스 본문에나 등장하지만 -> 중요한 단어는 아니다
- 문서의 핵심 단어 추출에 쓰임 -> 검색(BM25) 하이브리드 RAG 구축, 임베딩에 쓰이는 개념

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from kiwipiepy import Kiwi

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지(윈도우 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("../data/11-1_뉴스정제.csv")
kiwi = Kiwi()

In [18]:
# 명사 추출 함수 만들기
def extract_nouns(text):
    nouns = []

    # 토크나이저 적용
    result = kiwi.tokenize(text)
    for token in result:
        if token.tag.startswith("N"):   # N 으로 시작하는 글자가 명사
            nouns.append(token.form)    # 실제 명사만 넣기

    filtered_nouns = []
    for noun in nouns:
        if len(noun) > 1:
            filtered_nouns.append(noun)

    return filtered_nouns

### 1. TF-IDF 개념
- TF(단어 빈도) : 한 문서에서 그 단어가 자주 나올수록 높다
- IDF(역문서 빈도) : 그 단어가 여러 문서에 흔할수록 낮아진다.(흔한 단어에 패널티, 벌점)
- TF-IDF = TF x IDF : 이 문서에는 자주 나오지만, 다른 문서에는 드물수록 점수가 높다 

In [19]:
docs = ["자동차 가격 자동차",       # 문서 1
        "자동차 가격 가격",         # 문서 2
        "날씨 날씨 자동차"          # 문서 3
]

vocab = ['자동차', '가격', '날씨']

data = pd.DataFrame({
    "자동차" : [2, 1, 1],
    "가격" : [1, 2, 0],
    "날씨" : [0, 0, 2]
})
data

,자동차,가격,날씨
0,2,1,0
1,1,2,0
2,1,0,2


In [20]:
import numpy as np

# 전체 문서수
N = 3
df_ = (data > 0).sum()    # 각 단어가 등장한 문서 수
idf = np.log(N / df_)
idf

자동차    0.000000
가격     0.405465
날씨     1.098612
dtype: float64

In [21]:
# 사이킷런 패키지를 이용해서 tf-idf 구하기
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
df['정제본문']

0      서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1      전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2      NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3      img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4      파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...
                             ...                        
995    위성 전력 충전 상태 안정적으로 작동 지상국 명령 따라 정상 임무 확인 위성 안정적...
996    R D 비용 50억원 필라테스 사이클 동작 구현 가능 연내 생체신호 센서 탑재 제품...
997    직방이 궁극적으로 추구하는 방향은 라이프스타일 플랫폼이다 초창기에는 원룸 위주의 부...
998    요즘 스타트업의 비즈니스 모델 을 살펴봅니다 네이버와 카카오가 보여주는 스타트업 투...
999    7월 포켓배틀스 NFT War 출시 메인넷 미버스 생태계 구축한다 포켓배틀스 NFT...
Name: 정제본문, Length: 1000, dtype: str

In [23]:
vec = TfidfVectorizer(tokenizer=extract_nouns, token_pattern=None)
X = vec.fit_transform(df['정제본문'])

In [24]:
feature_words = vec.get_feature_names_out()
print(feature_words)

['1이더리움' '2여객터미널' '3사' ... '힘겨루기' '힙밥' '힙합']


In [25]:
print("문서 수  x 단어 수", X.shape)

문서 수  x 단어 수 (1000, 14444)


In [26]:
len(feature_words)

14444

In [27]:
# 한 기사에서 가장 tf-idf 값이 높은 단어를 찾아보는 함수를 만들어 봅시다
def keywords(docs_idx, count=5): # 문서의 넘버를 받아서 가장 높은 단어를 몇개 볼지 정해주는 함수
    row = X[docs_idx].toarray()[0]
    top = row.argsort()[-count:][::-1]
    return [feature_words[i] for i in top]

In [28]:
df[['제목', '정제본문']].head()

,제목,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


In [29]:
keywords(1)

['이스타항공', '회생', '의원', '관계', '오해']

In [30]:
keywords(3)

['재정부', '기획', '완화', '대출', '유류']

In [ ]:
pd.DataFrame(X.data.T)


### 문서별 단어 tf-idf 였는데
- 각 문서가 얼마나 유사한가?
- 코사인 유사도로 계산